In [1]:
from Bio.PDB import PDBParser
from rdkit import Chem

import numpy as np

import torch
import torch_cluster
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import GINEConv, knn_graph

/Users/martvanderlugt/Dev/VU/ML4CHEM/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load protein


In [2]:
parser = PDBParser()

pdb_code = '1a30'

structure = parser.get_structure("protein", f"./data/PDBBind/{pdb_code}/{pdb_code}_protein.pdb")
structure

<Structure id=protein>

In [3]:
AMINO_ACIDS = [
    'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
    'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL'
]
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

coords, features = [], []

for residue in structure.get_residues():
    # skip non aminoacids
    if residue.get_id()[0] != ' ':
        continue

    res_name = residue.get_resname()

    # skip unknown aminoacids
    if res_name not in AA_TO_IDX:
        continue

    ca = residue['CA'].get_vector().get_array()
    one_hot = np.zeros(len(AMINO_ACIDS))
    one_hot[AA_TO_IDX[res_name]] = 1.0

    coords.append(ca)
    features.append(one_hot)

coords = torch.tensor(np.array(coords), dtype=torch.float32)    # (N, 3)
features = torch.tensor(np.array(features), dtype=torch.float32)    # (N, 20)

coords.shape, features.shape

(torch.Size([198, 3]), torch.Size([198, 20]))

# Define model

In [4]:
class PocketFinder(nn.Module):
    def __init__(self, in_dim: int = 20, hidden: int = 64):
        super().__init__()

        # from one-hot encoding to embedding
        self.embedding = nn.Linear(in_dim, hidden)

        self.convs = nn.ModuleList(
            GINEConv(nn = nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.ReLU(),
                nn.Linear(hidden, hidden)
            ), edge_dim=1)
            for _ in range(3)
        )

        self.head = nn.Sequential(
            nn.Linear(hidden, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, h, pos, batch):
        h = self.embedding(h)

        edge_index = knn_graph(pos.cpu(), k=10, batch=batch.cpu(), loop=False)
        row, col = edge_index.to(h.device)
        edge_attr = (pos[row] - pos[col]).norm(dim=-1, keepdim=True)

        for conv in self.convs:
            h = h + conv(h, edge_index, edge_attr)

        h = self.head(h)

        return h.squeeze(-1)

In [5]:
model = PocketFinder(in_dim = len(AMINO_ACIDS))
model

PocketFinder(
  (embedding): Linear(in_features=20, out_features=64, bias=True)
  (convs): ModuleList(
    (0-2): 3 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (head): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

#### Test model (random data)

In [6]:
pos = torch.randn(50, 3)
h = torch.zeros(50, 20)
batch = torch.zeros(50, dtype=torch.long)

out = model(h, pos, batch)
print(out.shape)

torch.Size([50])


In [7]:
probs = model(h, pos, batch).sigmoid()
print(probs.shape)
print(probs.min().item(), probs.max().item())

torch.Size([50])
0.6244087219238281 0.6926631331443787


# Loading ligand


In [8]:
mol = Chem.MolFromMol2File(f'./data/PDBBind/{pdb_code}/{pdb_code}_ligand.mol2')
conf = mol.GetConformer()

ligand_coords = torch.tensor(conf.GetPositions(), dtype=torch.float32)  # (L, 3)

print(ligand_coords.shape)

torch.Size([26, 3])


# Generate pocket labels


In [9]:
POCKET_DIST_CUTOFF = 6.5

pairwise_dist_mat = torch.cdist(coords, ligand_coords)  # (N, L)
print(pairwise_dist_mat.shape)

N, L = pairwise_dist_mat.shape

pocket_labels = (pairwise_dist_mat.min(dim=1).values < POCKET_DIST_CUTOFF).float()  # (N,)
print(pocket_labels.sum().item(), pocket_labels.shape)


torch.Size([198, 26])
21.0 torch.Size([198])


#### Test model (single protein)

In [10]:
batch = torch.zeros(N)

logits = model(features, coords, batch)

print(logits.shape) # (N,)

torch.Size([198])


# Define Loss


In [11]:
loss = F.binary_cross_entropy_with_logits(logits, pocket_labels,
                                          pos_weight=torch.tensor(9))

print(loss.item())

2.892484188079834


# Training


In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

optimizer.zero_grad()
logits = model(features, coords, batch)
loss = F.binary_cross_entropy_with_logits(logits, pocket_labels,
                                          pos_weight=torch.tensor(9))
loss.backward()
optimizer.step()

print(loss.item())

2.892484188079834


In [13]:
with torch.no_grad():
    probs = model(features, coords, batch).sigmoid()

print("predicted pocket residues:", (probs > 0.5).sum().item())
print("actual pocket residues:", pocket_labels.sum().item())
print("max prob:", probs.max().item())
print("min prob:", probs.min().item())

predicted pocket residues: 198
actual pocket residues: 21.0
max prob: 0.9613995552062988
min prob: 0.9299672842025757


# scPDB


In [14]:
def parse_mol2_protein(path):
    coords, features = [], []
    in_atom_block = False
    seen_residues = {}  # res_id -> (resname, ca_coords)

    with open(path) as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>"):
                in_atom_block = False
                continue
            if not in_atom_block:
                continue

            parts = line.split()
            atom_name = parts[1]
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            res_id = parts[6]
            res_name = parts[7][:3].upper()

            if atom_name == "CA" and res_name in AA_TO_IDX:
                if res_id not in seen_residues:
                    seen_residues[res_id] = True
                    one_hot = np.zeros(20)
                    one_hot[AA_TO_IDX[res_name]] = 1.0
                    coords.append([x, y, z])
                    features.append(one_hot)

    pos = torch.tensor(np.array(coords), dtype=torch.float)
    h = torch.tensor(np.array(features), dtype=torch.float)
    return pos, h

In [15]:
def parse_mol2_coords(path):
    coords = []
    in_atom_block = False

    with open(path) as f:
        for line in f:
            if line.startswith("@<TRIPOS>ATOM"):
                in_atom_block = True
                continue
            if line.startswith("@<TRIPOS>"):
                in_atom_block = False
                continue
            if not in_atom_block:
                continue

            parts = line.split()
            x, y, z = float(parts[2]), float(parts[3]), float(parts[4])
            coords.append([x, y, z])

    return torch.tensor(np.array(coords), dtype=torch.float)


In [16]:
coords, features = parse_mol2_protein('./data/scPDB/1a2b_1/protein.mol2')
N = coords.shape[0]

coords.shape, features.shape

(torch.Size([178, 3]), torch.Size([178, 20]))

In [17]:
site_coords = parse_mol2_coords("data/scPDB/1a2b_1/site.mol2")
print(site_coords.shape)

torch.Size([496, 3])


In [18]:
cavity_coords = parse_mol2_coords("data/scPDB/1a2b_1/cavity6.mol2")
print(cavity_coords.shape)

torch.Size([120, 3])


In [19]:
for threshold in [0.5,  1, 2, 3, 4, 5]:
    labels = (torch.cdist(coords, site_coords).min(dim=1).values < threshold).float()
    print(f"{threshold}Å: {labels.sum().item():.0f} pocket residues ({100*labels.mean().item():.1f}%)")

0.5Å: 34 pocket residues (19.1%)
1Å: 34 pocket residues (19.1%)
2Å: 34 pocket residues (19.1%)
3Å: 44 pocket residues (24.7%)
4Å: 58 pocket residues (32.6%)
5Å: 67 pocket residues (37.6%)


# PyTorch Dataset


In [20]:
import random
from collections import defaultdict
from torch.utils.data import Dataset, Subset
from pathlib import Path

class scPDBDataset(Dataset):
    def __init__(self, root_dir: str, device: str = "cpu"):
        self.device = device
        self.entries = [
            d for d in Path(root_dir).iterdir()
            if (d / 'protein.mol2').exists() and (d / 'site.mol2').exists()
        ]

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        entry = self.entries[idx]
        pos, h = parse_mol2_protein(entry / 'protein.mol2')
        site_coords = parse_mol2_coords(entry / 'site.mol2')

        dists = torch.cdist(pos, site_coords)
        labels = (dists.min(dim=1).values < 1.0).float()

        return h.to(device), pos.to(device), labels.to(device)


def split_dataset(dataset, val_frac=0.1, test_frac=0.1, seed=42):
    random.seed(seed)

    # group indices by PDB ID to avoid leakage
    pdb_groups = defaultdict(list)
    for i, entry in enumerate(dataset.entries):
        pdb_id = entry.name[:4]
        pdb_groups[pdb_id].append(i)

    pdb_ids = list(pdb_groups.keys())
    random.shuffle(pdb_ids)

    n = len(pdb_ids)
    n_val  = int(val_frac * n)
    n_test = int(test_frac * n)

    test_ids  = set(pdb_ids[:n_test])
    val_ids   = set(pdb_ids[n_test:n_test + n_val])
    train_ids = set(pdb_ids[n_test + n_val:])

    train_indices = [i for pdb_id in train_ids for i in pdb_groups[pdb_id]]
    val_indices   = [i for pdb_id in val_ids   for i in pdb_groups[pdb_id]]
    test_indices  = [i for pdb_id in test_ids  for i in pdb_groups[pdb_id]]

    return (
        Subset(dataset, train_indices),
        Subset(dataset, val_indices),
        Subset(dataset, test_indices)
    )


# Training loop

In [21]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

device = 'cpu' # Some errors on gpu

mps


In [22]:
def evaluate(model, subset):
    model.eval()
    all_f1 = []
    with torch.no_grad():
        for h, pos, labels in subset:
            batch = torch.zeros(len(pos), dtype=torch.long)
            probs = model(h, pos, batch).sigmoid()
            preds = (probs > 0.5).float()
            tp = (preds * labels).sum()
            fp = (preds * (1 - labels)).sum()
            fn = ((1 - preds) * labels).sum()
            precision = tp / (tp + fp + 1e-8)
            recall    = tp / (tp + fn + 1e-8)
            f1 = 2 * precision * recall / (precision + recall + 1e-8)
            all_f1.append(f1.item())
    return sum(all_f1) / len(all_f1)

In [23]:
from torch.utils.data import DataLoader

dataset = scPDBDataset('./data/scPDB', device)
train_set, val_set, test_set = split_dataset(dataset, val_frac=0.1, test_frac=0.1, seed=42)

def collate_fn(batch):
    h, pos, labels = batch[0]
    batch_idx = torch.zeros(len(pos), dtype=torch.long)
    return h, pos, labels, batch_idx

train_loader = DataLoader(
    train_set,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
)

model = PocketFinder(in_dim = len(AMINO_ACIDS), hidden=128).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

E = 3

for epoch in range(E):
    total_loss = 0

    for i, (h, pos, labels, batch) in enumerate(train_loader):

        optimizer.zero_grad()
        logits = model(h, pos, batch)
        loss = F.binary_cross_entropy_with_logits(logits, labels, pos_weight=torch.tensor(6.0, device=device))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    val_f1 = evaluate(model, val_set)
    print(f"epoch {epoch} loss {total_loss/len(train_set):.4f} val_f1 {val_f1:.3f}")

epoch 0 loss 0.9209 val_f1 0.217
epoch 1 loss 0.8722 val_f1 0.266
epoch 2 loss 0.8387 val_f1 0.292


In [24]:
model.eval()
with torch.no_grad():
    h, pos, labels = test_set[0]
    batch = torch.zeros(len(pos), dtype=torch.long)
    logits = model(h, pos, batch)
    probs = logits.sigmoid()

print("predicted pocket:", (probs > 0.5).sum().item())
print("actual pocket:", labels.sum().item())

print("logits min:", logits.min().item()

print("logits max:", logits.max().item())
print("logits mean:", logits.mean().item())

preds = (probs > 0.5).float()
tp = (preds * labels).sum()
fp = (preds * (1 - labels)).sum()
fn = ((1 - preds) * labels).sum()
precision = tp / (tp + fp + 1e-8)
recall = tp / (tp + fn + 1e-8)
f1 = 2 * precision * recall / (precision + recall + 1e-8)
print(f"precision: {precision:.3f}, recall: {recall:.3f}, f1: {f1:.3f}")

predicted pocket: 50
actual pocket: 28.0
logits min: -2.9006476402282715
logits max: 1.0444374084472656
logits mean: -0.5795773267745972
precision: 0.440, recall: 0.786, f1: 0.564
